In [10]:
from pydantic import BaseModel, EmailStr, Field
from typing import Literal

class MessageRequests(BaseModel):
    email: EmailStr
    message: str = Field(
        min_length=3,
        max_length=5000
    )

class ProcessResult(BaseModel):
    status: Literal["success", "failure"]
    error: str | None = None

In [11]:
from enum import Enum

class Department(str, Enum):
    HR = "human-resources@example.com"
    IT = "it@example.com"
    KADRY = "kadry@example.com"
    HELP_DESK = "help-desk@example.com"
    OTHER = "other@example.com"


In [12]:
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    smtp_host:str = "mailhog"
    smtp_port:int = 1025
    ollama_host:str = "http://ollama:11434"
    model_name:str = "llama3.2:3b"
    sender_email:str = "routing-agent@example.com"

settings = Settings()

In [13]:
import smtplib
from email.message import EmailMessage

class MailService:
    def send_email(
            self,
            recipient:str,
            sender: str,
            message: str,
    )-> None:
        email = EmailMessage()

        email.set_content(message)
        email["From"] = settings.sender_email
        email["To"] = recipient
        email["Reply-To"] = sender
        email["Subject"] = "NewRequest-routed"
        with smtplib.SMTP(settings.smtp_host, settings.smtp_port) as server:
            server.send_message(email)


In [14]:
# plik app/tools.py

from pydantic import BaseModel, EmailStr
# from app.models import Department

class SendEmailInput(BaseModel):
    recipient: Department
    sender: EmailStr
    message: str

In [15]:
from langchain_core.tools import tool
from smtplib import SMTPException

mail_service = MailService()

@tool("send_email", args_schema=SendEmailInput)
def send_email(recipient:str,sender:str, message:str)-> ProcessResult:
    """Sends an email to a specified recipient with a sender and message body."""
    try:
        mail_service.send_email(
            recipient=recipient,
            sender=sender,
            message=message,
        )
        return ProcessResult(status="success")

    except SMTPException as e:
        # Specific mail transport failure
        # logger.error(f"SMTP error while sending email to {input_model.recipient.value}: {e}")
        return ProcessResult(status="failure", error=f"Mail transport error: {str(e)}")

    except AttributeError as e:
        # Code/Data access bug (e.g., recipient missing .value)
        # logger.error(f"Invalid input model structure: {e}")
        return ProcessResult(status="failure", error="Invalid input data structure")

    except Exception as e:
        # logger.exception(f"Unexpected error sending email to {input_model.recipient.value}")
        return ProcessResult(status="failure", error=str(e))


In [40]:
SYSTEM_PROMPT = """ You are an automated AI routing agent. Your task is to analyze incoming user requests and route them to the correct department email by invoking the routing tool.

DEPARTMENT ROUTING DIRECTIVES:

1. "it@example.com"
   - Computer or hardware malfunctions
   - Printer setup, hardware, or driver issues
   - Network, internet, or VPN connectivity issues
   - Software installations or troubleshooting

2. "help-desk@example.com"
   - Password reset requests
   - Account lockout or account access issues

3. "human-resources@example.com"
   - Recruitment, job applications, or hiring process
   - Vacation requests, PTO, and leave management

4. "kadry@example.com"
   - Payroll, salary, compensation, or wage inquiries
   - Employment contracts, tax documents, and legal HR paperwork

5. "other@example.com"
   - Use as a FALLBACK for any request that does not clearly fit the categories above, or if the request is ambiguous/unclear.

STRICT OPERATIONAL BOUNDARIES:
0. DONT ANSWER ANYTHING!, Just the email address.
1. ALWAYS CALL A TOOL: You must invoke the designated tool call for every request. Never generate a text response to the user.
2. EXACT VALUES: Only route to the exact email address strings listed above.
3. PRIMARY INTENT: If a request contains multiple topics, identify the primary issue. If no single primary issue can be determined, route to "other@example.com".
"""

In [27]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:3b", temperature=0.0)

In [53]:
type(llm)

langchain_ollama.chat_models.ChatOllama

In [47]:
from langchain.agents import create_agent

agent_executor = create_agent(
    model=llm,
    tools = [send_email],
    system_prompt=SYSTEM_PROMPT
)

In [48]:
email = "master-yoda@example.com"
message = "I am looking for a job, CV I will send"

In [54]:
from langchain.agents import create_agent

class RoutingAgent:
    def __init__(self, llm:ChatOllama, tools, system_prompt: str):

        self.agent_executor = create_agent(
            model=llm,
            tools=tools,
            system_prompt=system_prompt
        )

    def process(self, email: str, message: str):
        return self.agent_executor.invoke({
            "messages": [
                ("user", f"sender: {email}\nmessage: {message}")
            ]
        })

In [58]:
router = RoutingAgent(
    llm=llm,
    tools=[send_email],
    system_prompt=SYSTEM_PROMPT
)

response = router.process(
    email="master-yoda@example.com",
    message="I am looking for a job, CV I will send"
)

print(response["messages"][-1].content)

Sorry, Master Yoda, it seems like the email routing tool encountered an error. The email address "human-resources@example.com" is not recognized. Please try again with a different email address.

{"name": "send_email", "parameters": {"message":"I am looking for a job, CV I will send","recipient":"other@example.com","sender":"master-yoda@example.com"}}


In [57]:
from functools import lru_cache

@lru_cache()
def get_agent()->RoutingAgent:
    return RoutingAgent(llm, [send_email], SYSTEM_PROMPT)